# 🎬 Longform Video Factory

Automated faceless whiteboard animation YouTube video pipeline.

**Workflow:**
1. Setup & credentials
2. Select topic from Google Sheet
3. Run Deep Research (Gemini)
4. Generate script (Claude/Gemini)
5. ⏸️ **Review script**
6. Generate voiceover (Fish Audio / Qwen3-TTS)
7. Generate scene illustrations (Gemini Flash Image)
8. Assemble video (FFmpeg)
9. Generate thumbnails
10. Generate SEO metadata
11. ⏸️ **Review video & metadata**
12. Finalize & Save status to Google Sheets


## Cell 0: Setup & Install


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Clone or update the repo
import os
REPO_DIR = '/content/longform'
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/techafreshh/longform.git {REPO_DIR}

# Install dependencies
!pip install -q google-genai fish-audio-sdk openai gspread google-auth \
    python-dotenv requests Pillow openai-whisper

# Add repo to path
import sys
sys.path.insert(0, REPO_DIR)

print('✅ Setup complete!')


## Cell 1: Credentials & API Keys

API keys are automatically loaded from Google Colab's Secrets manager (`google.colab.userdata`). Alternatively, you can uncomment Option B and set them manually.


In [ ]:
# Configure API keys
import os

# Option A: Import secrets automatically using google.colab.userdata
try:
    from google.colab import userdata
    for key in ['GOOGLE_API_KEY', 'OPENROUTER_API_KEY', 'FISH_API_KEY', 'FISH_VOICE_ID', 'PEXELS_API_KEY', 'GOOGLE_SHEET_ID', 'USE_VERTEX', 'GCP_PROJECT', 'GCP_LOCATION']:
        try:
            val = userdata.get(key)
            if val:
                os.environ[key] = str(val)
        except Exception:
            pass
    print('✅ Attempted importing secrets from Colab Userdata.')
except ImportError:
    print('ℹ️ Colab userdata not available. Using manually set environment variables.')

# Option B: Set directly (uncomment and fill if not using Colab Secrets)
# os.environ['GOOGLE_API_KEY'] = 'AIzaSy...'
# os.environ['OPENROUTER_API_KEY'] = 'sk-or-v1...'
# os.environ['FISH_API_KEY'] = '...'
# os.environ['FISH_VOICE_ID'] = '...'
# os.environ['PEXELS_API_KEY'] = '...'
# os.environ['GOOGLE_SHEET_ID'] = '...'
# Vertex AI options (uncomment to override Colab Secrets)
# os.environ['USE_VERTEX'] = 'true'
# os.environ['GCP_PROJECT'] = 'your-gcp-project-id'
# os.environ['GCP_LOCATION'] = 'us-central1'

# Authenticate with Google (for Sheets access)
try:
    from google.colab import auth
    auth.authenticate_user()
    print('✅ Authenticated with Google Account.')
except ImportError:
    print('ℹ️ Colab auth not available (running locally).')


## Cell 2: Fetch Topics

Fetches all topics marked as `ready` in your Google Sheet.


In [ ]:
from src.sheets import get_ready_topics, update_status

# Fetch ready topics from your Google Sheet
try:
    topics = get_ready_topics()
    if not topics:
        print('⚠️  No topics with status "ready" found in the sheet.')
        print('   Add topics to your sheet or set one manually below.')
    else:
        print('\nSelect a topic by number:')
        for i, t in enumerate(topics):
            print(f'  [{i}] {t["topic"]} ({t["niche"]}, {t["style"]})')
except Exception as e:
    print(f'⚠️  Failed to load topics from Google Sheet: {e}')
    print('   You can define a topic manually in the next cell.')


## Cell 3: Pick Topic

Configure which topic to run (using the index from above or manual entry).


In [ ]:
#=== Pick your topic ===#
# Option A: From sheet (set the index from above)
TOPIC_INDEX = 0  # Change this to select a different topic

try:
    selected = topics[TOPIC_INDEX]
    TOPIC = selected['topic']
    NICHE = selected['niche']
    STYLE = selected['style']
    ADDITIONAL_PROMPT = selected['additional_prompt']
    TARGET_LENGTH = selected['target_length']
    SHEET_ROW = selected['row_number']
    print(f'✅ Selected from Sheet: "{TOPIC}"')
except (NameError, IndexError):
    # Option B: Manual entry
    TOPIC = "Why You Can\'t Remember Being a Baby"
    NICHE = "Psychology"
    STYLE = "color_whiteboard"  # or "chalkboard"
    ADDITIONAL_PROMPT = "Focus on hippocampus development"
    TARGET_LENGTH = "10-12 min"
    SHEET_ROW = None
    print(f'✅ Manual topic: "{TOPIC}"')

BASE_DIR = '/content/drive/MyDrive/LongformFactory'
print(f'   Output folder will be: {BASE_DIR}/{NICHE}/{TOPIC[:40]}...')


## Cell 4: Initialize Paths

Sets up directory structures for the project.


In [ ]:
from pathlib import Path
from src.config import ProjectPaths, slugify

# Set up project path structure
paths = ProjectPaths(
    base_dir=Path(BASE_DIR),
    niche=slugify(NICHE),
    topic_slug=slugify(TOPIC),
)
paths.ensure_dirs()
print(f'✅ Project paths initialized at: {paths.project_dir}')


## Cell 5: Stage 1 - Deep Research

Uses Gemini Deep Research to write a research document. If the research file already exists, it is loaded from disk to save credits.


In [ ]:
from src.pipeline import run_stage_research

# Force regeneration: set to True if you want to overwrite existing research
FORCE_RESEARCH = False

if SHEET_ROW:
    update_status(SHEET_ROW, 'researching')

research_text = run_stage_research(
    paths=paths,
    topic=TOPIC,
    niche=NICHE,
    additional_prompt=ADDITIONAL_PROMPT,
    force=FORCE_RESEARCH,
)


## Cell 6: Stage 2 - Script Generation

Generates the narration script with embedded scene markers. You can change the model here (e.g. `anthropic/claude-3-5-sonnet`). If the script already exists, it is loaded from disk to save credits.


In [ ]:
from src.pipeline import run_stage_script

# Configure scriptwriting model. Default is Claude Sonnet via OpenRouter.
SCRIPT_MODEL = 'anthropic/claude-3-5-sonnet'
FORCE_SCRIPT = False

script_obj = run_stage_script(
    paths=paths,
    topic=TOPIC,
    niche=NICHE,
    research_text=research_text,
    style=STYLE,
    target_length=TARGET_LENGTH,
    additional_prompt=ADDITIONAL_PROMPT,
    model=SCRIPT_MODEL,
    force=FORCE_SCRIPT,
)

if SHEET_ROW:
    update_status(SHEET_ROW, 'review_script')

print(f'✅ Script generation complete!')
print(f'   Scene count: {script_obj.scene_count}')


## Cell 7: ⏸️ Script Review Gate

Review the generated script below. If you want to make manual changes, open the script file on Drive, edit it, and save it. If you want to regenerate it entirely, change settings in Cell 6 and re-run Cell 6.


In [ ]:
from IPython.display import Markdown, display

# Display the script for manual review
if paths.script_file.exists():
    script_text = paths.script_file.read_text(encoding='utf-8')
    display(Markdown(script_text))
else:
    print('⚠️ Script file not found')

print('\n' + '='*60)
print('👆 Review the script above.')
print('   If you want to edit it, open the file on Google Drive and save it.')
print('   If you want to regenerate it, change SCRIPT_MODEL/prompts in Cell 6 and re-run.')


## Cell 8: Stage 3 - Voice Generation

Generates the voiceover track. If the voiceover already exists, it is loaded from disk to save credits.


In [ ]:
from src.pipeline import run_stage_voice

# TTS Engine settings
TTS_ENGINE = 'fish'           # 'fish', 'qwen_1.7b', or 'qwen_0.6b'
VOICE_MODEL = 's2.1-pro-free' # 's2.1-pro-free' or other Fish model version
REFERENCE_AUDIO = None        # Reference audio path for Qwen cloning
FORCE_VOICE = False

if SHEET_ROW:
    update_status(SHEET_ROW, 'generating')

scenes_data = [s.to_dict() for s in script_obj.scenes]

voice_result = run_stage_voice(
    paths=paths,
    scenes_data=scenes_data,
    tts_engine=TTS_ENGINE,
    voice_model=VOICE_MODEL,
    reference_audio=REFERENCE_AUDIO,
    force=FORCE_VOICE,
)

print(f'✅ Voiceover duration: {voice_result.total_duration:.1f}s')


## Cell 9: Stage 4 - Scene Image Generation

Generates the whiteboard/chalkboard illustration for each scene. If the images already exist in the folder, it is skipped to save credits.


In [ ]:
from src.pipeline import run_stage_scenes

FORCE_SCENES = False

image_paths = run_stage_scenes(
    paths=paths,
    scenes_data=scenes_data,
    style=STYLE,
    force=FORCE_SCENES,
)

print(f'✅ Generated/Found {len(image_paths)} scene images.')


## Cell 10: Stage 5 - Video Assembly

Assembles the final video using FFmpeg, applying Ken Burns and transitions, mixing voiceover and BGM, and burning subtitles. If the final video already exists, it is skipped.


In [ ]:
from src.pipeline import run_stage_assembly

BGM_PATH = None       # Path to background music (or None)
BGM_VOLUME = 0.15     # BGM volume (0-1)
KEN_BURNS = True      # Apply zoom/pan animation
FORCE_ASSEMBLY = False

final_video_path = run_stage_assembly(
    paths=paths,
    scenes_data=scenes_data,
    voice_result=voice_result,
    style=STYLE,
    bgm_path=BGM_PATH,
    bgm_volume=BGM_VOLUME,
    ken_burns=KEN_BURNS,
    force=FORCE_ASSEMBLY,
)

print(f'✅ Video assembled at: {final_video_path}')


## Cell 11: Stage 6 - Thumbnail Generation

Generates 3 YouTube thumbnail options. Skipped if they already exist.


In [ ]:
from src.pipeline import run_stage_thumbnails

FORCE_THUMBNAILS = False

thumbnail_paths = run_stage_thumbnails(
    paths=paths,
    topic=TOPIC,
    niche=NICHE,
    style=STYLE,
    force=FORCE_THUMBNAILS,
)

print(f'✅ Generated/Found {len(thumbnail_paths)} thumbnail variants.')


## Cell 12: Stage 7 - SEO Metadata

Generates YouTube title, tags, description, and category. Skipped if it already exists.


In [ ]:
from src.pipeline import run_stage_seo

FORCE_SEO = False

seo_data = run_stage_seo(
    paths=paths,
    topic=TOPIC,
    niche=NICHE,
    script_text=script_obj.raw_text,
    style=STYLE,
    force=FORCE_SEO,
)

print(f'✅ SEO Metadata generated!')


## Cell 13: ⏸️ Video & Thumbnail Review Gate

Watch the preview of the video, thumbnails, and SEO text before publishing.


In [ ]:
from IPython.display import Video, display, Image
from pathlib import Path

# Preview the generated video
if final_video_path.exists():
    print(f'🎬 Video path: {final_video_path}')
    print(f'   Duration: {voice_result.total_duration:.1f}s')
    
    try:
        display(Video(str(final_video_path), embed=True, width=800))
    except Exception:
        print('   (Preview player not available - check file on Google Drive)')
else:
    print('⚠️ Video file not found')

# Preview the thumbnails
if thumbnail_paths:
    print('\n🖼️ Thumbnail variants:')
    for tp in thumbnail_paths:
        if tp.exists():
            display(Image(filename=str(tp), width=400))

# Preview the SEO metadata
print('\n📊 SEO Metadata:')
print(f'   Title: {seo_data.get("title", "N/A")}')
print(f'   Description: {seo_data.get("description", "N/A")[:200]}...')
print(f'   Tags: {json.dumps(seo_data.get("tags", []))}')


## Cell 14: Finalize & Mark Complete

Mark the video as complete in the Google Sheet and upload path metadata.


In [ ]:
from src.sheets import update_status

if SHEET_ROW:
    drive_link = str(final_video_path)
    thumbnail_link = str(thumbnail_paths[0]) if thumbnail_paths else ""
    
    update_status(
        SHEET_ROW,
        'complete',
        extra_data={
            'drive_link': drive_link,
            'thumbnail_link': thumbnail_link,
            'seo_title': seo_data.get('title', ''),
            'seo_description': seo_data.get('description', ''),
            'seo_tags': ', '.join(seo_data.get('tags', [])) if isinstance(seo_data.get('tags'), list) else seo_data.get('tags', '')
        }
    )
    print("✅ Spreadsheet updated and topic marked 'complete'!")
else:
    print("ℹ️ Run was started manually. Output saved to Drive but Sheet not updated.")


## 🛠️ Utilities


In [ ]:
# === UTILITY: Clone your voice on Fish Audio ===
# Run this ONCE with your voice sample, then save the voice_id

# from src.voice import clone_voice_fish
# voice_id = clone_voice_fish(
#     audio_path='/content/drive/MyDrive/my_voice_sample.wav',
#     voice_name='my-narrator-voice',
# )
# print(f'Set this in your config: FISH_VOICE_ID={voice_id}')


In [ ]:
# === UTILITY: Initialize Google Sheet template ===
# Run this ONCE to set up your content calendar headers

# from src.sheets import create_sheet_template
# create_sheet_template()
